# Bernett res_bidirect pair-order check

Goal: compare prediction for `(p1, p2)` with `(p2, p1)` for the requested checkpoint.

Requested checkpoint/eval folder:
`/hpc/home/wl324/projects/tt3d/data/results/bernett_esm2_train_res_bidirect_lr0.00025_wd0.0002_dp0.5_grid32_d8_sym1.0/eval-bernett_res_bidirect_best_model`

Original test TSV:
`/hpc/home/wl324/projects/tt3d/data_archive/bernett_test.tsv`


In [ ]:
from pathlib import Path

root = Path('/hpc/home/wl324/D-SCRIPT')
model = Path('/hpc/home/wl324/projects/tt3d/data/results/bernett_esm2_train_res_bidirect_lr0.00025_wd0.0002_dp0.5_grid32_d8_sym1.0/bernett_res_bidirect_best_model.sav')
embeddings = Path('/hpc/home/wl324/projects/tt3d/data_archive/esm2/bernett')
test = Path('/hpc/home/wl324/projects/tt3d/data_archive/bernett_test.tsv')
original_predictions = Path('/hpc/home/wl324/projects/tt3d/data/results/bernett_esm2_train_res_bidirect_lr0.00025_wd0.0002_dp0.5_grid32_d8_sym1.0/eval-bernett_res_bidirect_best_model/bernett_bernett_res_bidirect_best_model.sav.predictions.tsv')
swapped_test = root / 'temp/bernett_test_swapped.tsv'
probe_out = root / 'temp/bernett_res_bidirect_swap_check.tsv'

model, embeddings, test, original_predictions, swapped_test, probe_out


In [ ]:
# Create the swapped test file: p1<->p2, label preserved.
import pandas as pd

df = pd.read_csv(test, sep='\t', header=None)
df[[1, 0, 2]].to_csv(swapped_test, sep='\t', header=False, index=False)
len(df), swapped_test


The full swapped Slurm job script is saved at `temp/test_res_bidirect_bernett_swapped.sh`. In this interactive shell, Slurm commands (`sbatch`, `sinfo`, `squeue`) did not return, and the foreground evaluator saw `torch.cuda.is_available() == False`, so a full CPU run would take multiple hours.

For the symmetry question, one mismatch is sufficient. The probe below computes both directions with the same checkpoint for the first 100 Bernett test pairs.

In [ ]:
!/hpc/home/wl324/projects/tt3d/data_archive/env/dscript/bin/python temp/check_res_bidirect_swap_symmetry.py \
  --model /hpc/home/wl324/projects/tt3d/data/results/bernett_esm2_train_res_bidirect_lr0.00025_wd0.0002_dp0.5_grid32_d8_sym1.0/bernett_res_bidirect_best_model.sav \
  --test /hpc/home/wl324/projects/tt3d/data_archive/bernett_test.tsv \
  --embeddings /hpc/home/wl324/projects/tt3d/data_archive/esm2/bernett \
  --limit 100 \
  --outfile temp/bernett_res_bidirect_swap_check.tsv \
  -d 0


In [ ]:
check = pd.read_csv(probe_out, sep='\t')
summary = {
    'rows': len(check),
    'exact_equal_full_float': int((check.pred_p1_p2 == check.pred_p2_p1).sum()),
    'equal_after_5_decimal_rounding': int((check.pred_p1_p2.round(5) == check.pred_p2_p1.round(5)).sum()),
    'different_after_5_decimal_rounding': int((check.pred_p1_p2.round(5) != check.pred_p2_p1.round(5)).sum()),
    'min_abs_delta': check.abs_delta.min(),
    'mean_abs_delta': check.abs_delta.mean(),
    'median_abs_delta': check.abs_delta.median(),
    'max_abs_delta': check.abs_delta.max(),
}
summary


In [ ]:
check.sort_values('abs_delta', ascending=False).head(5)


Observed result from the completed 100-pair probe: predictions are not the same after swapping the pair order. All 100 tested pairs differed even after 5-decimal rounding. The largest observed absolute difference was `0.0258717537` for `P15941, Q969S0`: `0.610538` vs `0.584666`.

## Why the model is order-sensitive

The difference is caused by order-sensitive parts of `dscript/models/interaction_res_bidirect.py`. The model has separate side-0 and side-1 modules (`sa0`/`sa1`, `ln0_*`/`ln1_*`, `ff0`/`ff1`, `pool0`/`pool1`) defined around lines 392-405 and used around lines 491-509. When we swap `(p1, p2)` to `(p2, p1)`, each protein goes through a different learned pathway.

The model also creates an explicitly antisymmetric feature at line 514: `int_sub = p0 - p1`. After swapping the pair, this becomes `p1 - p0`, which is the negative of the original value. That feature is concatenated into the classifier input at line 597, so the classifier can produce different logits for the two directions.

The contact-map path is not explicitly symmetrized either: `C` is built from `(contact_e0, contact_e1)` around line 482 and passed to `self.clf(yhat, g)` around line 598. Unless that path is designed to be transpose-invariant, it can also contribute to order dependence.

For ordinary PPI prediction, the target relation is undirected, so I would make the reported prediction symmetric. For this existing checkpoint, the safest immediate fix is inference-time averaging: `score_sym(p1, p2) = 0.5 * (score(p1, p2) + score(p2, p1))`. For a future model, retrain with a symmetric architecture: share the side-specific modules or use one common stack, remove `int_sub` or keep only symmetric pair features such as `p0 + p1`, `p0 * p1`, and `abs(p0 - p1)`, and symmetrize the contact-map branch. A consistency loss can help, but architecture or inference averaging is needed if exact equality is required.